# Project - AI for Medical Diagnosis and Prediction | Week #4

In this notebook, we continue our analysis of the MIMIC-CXR dataset by training convolutional neural networks for pathology classification. The objective is to perform a benchmark analysis of several models to detect the presence of pathologies in chest x-rays. 

In addition, we will train a segmentation model with a classification head. The objective is to see if the segmentation of relevant areas helps for pathology detection.

We will use a subset of the **MIMIC-CXR dataset** **[1][2]**. The MIMIC Chest X-ray (MIMIC-CXR) Database v2.0.0 is a large, publicly available dataset of chest radiographs in DICOM format, accompanied by free-text radiology reports. It contains 377,110 images from 227,835 radiographic studies conducted at the Beth Israel Deaconess Medical Center in Boston, MA. The dataset has been de-identified in compliance with the US Health Insurance Portability and Accountability Act of 1996 (HIPAA) Safe Harbor requirements. All protected health information (PHI) has been removed. More details: [https://mimic.mit.edu/docs/iv/modules/cxr/](https://mimic.mit.edu/docs/iv/modules/cxr/)

<div class="alert alert-block alert-info">
<b>Your tasks are the following:</b>  <br>
- Load the dataset using the same train, validation and test splits as during the previous week <i>(Task 1)</i> <br>
- Implement one (or two) convolutional neural networks (pretrained) and fine-tune them on your dataset <i>(Task 2)</i> <br>
- Evaluate the performance of the models using appropriate metrics <i>(Task 3)</i> <br>
- Implement a segmentation network of your choice (pretrained) <i>(Task 4)</i> <br>
- Add a classification head in addition to the segmentation network and train it on your dataset <i>(Task 4*)</i> <br>
- Evaluate the performance of the model, save the results of both models in a csv file using various appropriate metrics <i>(Task 4)</i> <br>
- Save the models and comment on their performance on the validation set <i>(Task 5)</i> <br>
- Compute the performance of the best model on the test set <i>(Task 5*)</i> <br>
</div>

**[1]** Johnson, A., Pollard, T., Mark, R., Berkowitz, S., & Horng, S. (2024). MIMIC-CXR Database (version 2.1.0). PhysioNet. [https://doi.org/10.13026/4jqj-jw95](https://doi.org/10.13026/4jqj-jw95).

**[2]** Johnson, A.E.W., Pollard, T.J., Berkowitz, S.J. et al. MIMIC-CXR, a de-identified publicly available database of chest radiographs with free-text reports. Sci Data 6, 317 (2019). [https://doi.org/10.1038/s41597-019-0322-0](https://doi.org/10.1038/s41597-019-0322-0)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

!pip install pydicom
!python3.8 -m pip install opencv-python
import pydicom
import time
import cv2
from PIL import Image
import re
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torchvision import tv_tensors
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score
from sklearn.metrics import RocCurveDisplay
from sklearn.metrics import roc_auc_score

torch.mps.empty_cache()

In [ ]:
DATA_PATH = '../data/MIMIC-CXR'

If you do not have the dataset anymore, please re-run the following cell to download it. Here, we will use the chest X-ray images and the labels extracted. 

In [ ]:
# !wget https://uni-bonn.sciebo.de/s/Rb66iDHGPrJiRAq/download --output {DATA_PATH}

In [ ]:
labels_df = pd.read_csv('labels.csv')

In [ ]:
labels_df.head()

## Task 1 - Dataset 
* Retrieve the train and test sets of patients used during week 3. Note that this is important to re-use the same splits for a fair comparison.
* Complete the `ClassificationDataset` class: you should be able to get images and corresponding labels using the dataframe and image directory path.
* Complete the `SegmentationDataset` class: you should be able to get images, corresponding masks and labels using the dataframe, image directory path and mask directory path.
* Split the train dataset to have a validation set for hyperparameter optimization and training followup.

In [ ]:
train_df = pd.read_csv('train_labels.csv')
patient_train = train_df['subject_id'].unique()

patient_train, patient_val = ... # COMPLETE

train_df = labels_df.loc[labels_df['subject_id'].isin(patient_train)]
val_df = labels_df.loc[labels_df['subject_id'].isin(patient_val)]

In [ ]:
class ClassificationDataset(Dataset):
    def __init__(self, dataframe, image_dir, train=False):
        super().__init__()
        self.df = dataframe
        self.image_ids = self.df['dicom_id'].tolist()
        self.study_ids = self.df['study_id'].tolist()
        self.subject_ids = self.df['subject_id'].tolist()
        self.labels = self.df['pathology'].tolist()
        self.image_dir = image_dir
        self.size = 512

        if train:
            self.transforms_img = torchvision.transforms.Compose([
                torchvision.transforms.Resize((self.size, self.size)),
                torchvision.transforms.RandomAdjustSharpness(sharpness_factor=2),
                torchvision.transforms.ToTensor(),
                torchvision.transforms.Normalize((0.5,), (0.2,))
            ])
        else: 
            self.transforms_img = torchvision.transforms.Compose([
                torchvision.transforms.Resize((self.size, self.size)),
                torchvision.transforms.ToTensor(),
                torchvision.transforms.Normalize((0.5,), (0.2,))
            ])

    def __getitem__(self, index: int):
        image_id = ...
        study_id = ... 
        subject_id = ...
        target =...
        
        image_path = f'{self.image_dir}/p{subject_id}/s{study_id}/{image_id}.dcm'
        image_dicom = ...
        image_array = ...
        image_array = (image_array / image_array.max() * 255).astype(np.uint8)
        image = Image.fromarray(image_array, mode = "L")
        image = self.transforms_img(image)

        target = torch.tensor(target)

        return image, target

    def __len__(self) -> int:
        return len(self.image_ids)

In [ ]:
train_dataset_cls = ClassificationDataset(
    train_df, 
    f'{DATA_PATH}/files/p18',
    train = True
)

val_dataset_cls = ClassificationDataset(
    val_df, 
    f'{DATA_PATH}/files/p18'
)

In [ ]:
# Plot a few examples
train_dataloader = DataLoader(train_dataset_cls, batch_size=32, shuffle=True)

f,ax = plt.subplots(1,2, figsize=(10,5))

for X, y in train_dataloader:
    first_zero = (y == 0).nonzero(as_tuple=True)[0][0].item()
    img = X[first_zero].cpu().numpy()
    img = np.transpose(img, (1,2,0))
    ax[0].imshow(img)
    ax[0].set_title('Label:' + str(y[first_zero]))

    if (y == 1).nonzero(as_tuple=False).numel() > 0:
        first_one = (y == 1).nonzero(as_tuple=True)[0][0].item()
        img = X[first_one].cpu().numpy()
        img = np.transpose(img, (1,2,0))
        ax[1].imshow(img)
        ax[1].set_title('Label:' + str(y[first_one]))
        break

In [ ]:
class SegmentationDataset(Dataset):
    def __init__(self, dataframe, image_dir, mask_dir, trs=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.df = dataframe

        self.image_ids = ... # COMPLETE
        self.study_ids = ... # COMPLETE
        self.subject_ids = ... # COMPLETE
        self.labels = ... # COMPLETE

        self.size = 256
        self.transforms_img = torchvision.transforms.Compose([
                torchvision.transforms.Resize((self.size, self.size)),
                torchvision.transforms.ToTensor(),
                torchvision.transforms.Normalize((0.5,), (0.2,))
            ])


        self.transforms_mask = transforms.Compose([
            torchvision.transforms.Resize((self.size, self.size), interpolation=transforms.InterpolationMode.NEAREST),
            torchvision.transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        image_id = self.image_ids[index]
        study_id = self.study_ids[index]
        subject_id = self.subject_ids[index]
        target = self.labels[index]
        
        image_path = f'{self.image_dir}/p{subject_id}/s{study_id}/{image_id}.dcm'
        image_dicom = ... # COMPLETE
        image_array = ... # COMPLETE
        image_array = (image_array / image_array.max() * 255).astype(np.uint8)
        image = Image.fromarray(image_array, mode = "L")
        image = self.transforms_img(image)

        target = torch.tensor(target)

        mask_path = f'{self.mask_dir}/{subject_id}/{study_id}/{image_id}.png'
        mask = Image.open(mask_path)
        mask = self.transforms_mask(mask)

        return image, mask, target

In [ ]:
train_dataset_seg = SegmentationDataset(
    train_df, 
    f'{DATA_PATH}/files/p18',
    f'{DATA_PATH}/segmentation'
)

val_dataset_seg = SegmentationDataset(
    val_df, 
    f'{DATA_PATH}/files/p18',
    f'{DATA_PATH}/segmentation'
)

## Task 2 - Classification
* Implement a Convolutional Neural Network for pathology classification. You can choose your preferred one among those explored during the lectures.
* Complete the training and validation loop, and train the model on the training set.
* (Optionally) Optimize hyperparameters or model architecture using this validation set.

In [ ]:
# train on the GPU or on the CPU, if a GPU is not available
device = torch.device('mps') if torch.backends.mps.is_available() else torch.device('cpu')

def train_loop_cls(dataloader, model, loss_fn, optimizer, device):
    size = len(dataloader.dataset)
    # Set the model to training mode
    model.train()

    for batch, (X, y) in enumerate(dataloader):
        optimizer. ... # COMPLETE
        X, y = X.to(device), y.to(device)
        # Compute prediction and loss
        pred = model(X)
        pred = nn.functional.sigmoid(pred)
        loss = ... # COMPLETE

        # Backpropagation
        loss.backward()
        optimizer.step()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

    return model

def val_loop_cls(dataloader, model, loss_fn, device):
    # Set the model to evaluation mode
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = ... # COMPLETE
            pred = nn.functional.sigmoid(pred)
            
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Validation: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

    return correct

In [ ]:
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

torch.mps.empty_cache()

learning_rate = ... # COMPLETE
batch_size = ... # COMPLETE
epochs = ... # COMPLETE

model_enb0 = efficientnet_b0(weights='DEFAULT')
model_enb0.classifier[1] = nn.Linear(1280, 2)
model_enb0.features[0][0] = nn.Conv2d(1, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)

train_dataloader = ... # COMPLETE
val_dataloader = ... # COMPLETE

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_enb0.parameters(), lr=learning_rate) # torch.optim.SGD(model_enb0.parameters(), lr = learning_rate)#
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10, eta_min=0)

model_enb0.to(device)
loss_fn.to(device)

s = time.process_time() # start time

best_acc = 0
best_model = None

for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    model_enb0 = train_loop_cls(train_dataloader, model_enb0, loss_fn, optimizer, device)
    lr_scheduler.step()
    test_acc = val_loop_cls(val_dataloader, model_enb0, loss_fn, device)
    
    if test_acc > best_acc:
        best_acc = test_acc
        best_model = model_enb0
        
print("Done!")

torch.save(best_model.state_dict(), './efficientnetb0-pathology-cxr.pth')

e = time.process_time() # end time
print(e - s, "seconds")

## Task 3 - Segmentation + Classification 
* Implement a Convolutional Neural Network for segmentation. Add a classification head on top of it to perform pathology classification.
* Train the model, and observe the training and validation loss across epochs.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class UNet(nn.Module):
    def __init__(self, num_classes_cls=2, in_channels=3, out_channels=3):
        super().__init__()
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.maxpool = nn.MaxPool2d(2)
        self.classifier = nn.Linear(512, num_classes_cls)

        # Encoder
        self.enc1 = self.conv_block(in_channels, 64)
        self.enc2 = self.conv_block(64, 128)
        self.enc3 = self.conv_block(128, 256)

        # Bottleneck
        self.bottleneck = self.conv_block(256, 512)

        # Decoder
        self.dec3 = self.conv_block(512 + 256, 256)
        self.dec2 = self.conv_block(256 + 128, 128)
        self.dec1 = self.conv_block(128 + 64, 64)
        self.final = nn.Conv2d(64, out_channels, kernel_size=1)

    def conv_block(self, in_c, out_c):
        return nn.Sequential(
            nn.Conv2d(in_c, out_c, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )

    def encode(self, x):
        # Encoder
        self.e1 = self.enc1(x)
        self.e2 = self.enc2(F.max_pool2d(self.e1, 2))
        self.e3 = self.enc3(F.max_pool2d(self.e2, 2))
        self.b = self.bottleneck(F.max_pool2d(self.e3, 2))

        return self.b

    def classify(self, b):
        # Classification
        cls_feat = self.avgpool(b).view(b.size(0), -1)
        class_logits = self.classifier(cls_feat)

        return class_logits

    def decode(self, b):
        # Decoder
        self.d3 = self.dec3(torch.cat([F.interpolate(b, scale_factor=2, mode='bilinear', align_corners=True), self.e3], dim=1))
        self.d2 = self.dec2(torch.cat([F.interpolate(self.d3, scale_factor=2, mode='bilinear', align_corners=True), self.e2], dim=1))
        self.d1 = self.dec1(torch.cat([F.interpolate(self.d2, scale_factor=2, mode='bilinear', align_corners=True), self.e1], dim=1))

        out = self.final(self.d1)

        return out

    def forward(self, x):
        b = self.encode(x)

        cls = self.classify(b)
        seg = self.decode(b)

        return cls, seg

In [ ]:
def train_loop_seg(model, dataloader, optimizer, criterion_seg, criterion_cls, device):
    model.train()
    running_loss = 0.0
    
    for images, masks, targets in dataloader:
        optimizer. ... # COMPLETE
        images = images.to(device)
        masks = masks.to(device)
        targets = targets.to(device)

        output_cls, output_seg = ... # COMPLETE
        output_cls = nn.functional.sigmoid(output_cls)
        loss_seg = criterion_seg(output_seg.float(), masks.float())
        loss_cls = criterion_cls(output_cls, targets)

        loss = 0.2 * loss_seg + 0.8 * loss_cls

        loss. ... # COMPLETE
        optimizer. ... # COMPLETE

        running_loss += loss.item()

    print(f'Loss: {running_loss / len(dataloader):.7f}')

    return model

def val_loop_seg(model, dataloader, criterion_seg, criterion_cls, device):
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    running_loss_cls = 0.0
    running_loss_seg = 0.0
    correct = 0
    
    for images, masks, targets in dataloader:
        
        images = images.to(device)
        masks = masks.to(device)
        targets = targets.to(device)

        output_cls, output_seg = ... # COMPLETE
        output_cls = nn.functional.sigmoid(output_cls)
        loss_seg = ... # COMPLETE
        loss_cls = ... # COMPLETE

        running_loss_seg += loss_seg.item()
        running_loss_cls += loss_cls.item()

        correct += (output_cls.argmax(1) == targets).type(torch.float).sum().item()

    correct /= size
    running_loss_seg /= num_batches
    running_loss_cls /= num_batches
    print(f'Validation\nClassification: {running_loss_cls / len(dataloader):.7f} (Accuracy: {(100*correct):.2f}%) \nSegmentation: {running_loss_seg / len(dataloader):.7f}')

    return running_loss_cls

In [ ]:
# train on the GPU or on the CPU, if a GPU is not available
device = torch.device('mps') if torch.backends.mps.is_available() else torch.device('cpu')
torch.mps.empty_cache()

learning_rate = ... # COMPLETE
batch_size = ... # COMPLETE
epochs = ... # COMPLETE

train_dataset_seg = SegmentationDataset(
    train_df, 
    f'{DATA_PATH}/files/p18',
    f'{DATA_PATH}/segmentation'
)

val_dataset_seg = SegmentationDataset(
    val_df, 
    f'{DATA_PATH}/files/p18',
    f'{DATA_PATH}/segmentation'
)

model = UNet(2,1,1)

# Define loss function and optimizer
criterion_seg = nn.BCEWithLogitsLoss()
criterion_cls = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

train_dataloader = ... # COMPLETE
val_dataloader = ... # COMPLETE

model.to(device)
best_acc = 100
best_model = None

for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    model = train_loop_seg(model, train_dataloader, optimizer, criterion_seg, criterion_cls, device)
    val_loss = val_loop_seg(model, val_dataloader, criterion_seg, criterion_cls, device)

    if val_loss < best_loss:
        best_loss=val_loss
        best_model = model
        
print("Done!")
torch.save(best_model.state_dict(), 'unet_seg_cls_pathology_cxr.pth')

## Task 4 - Evaluation 
* Implement a few evaluation metrics to compare the performance of the two networks. Remember that the objective is pathology detection, so you must take into account the importance of false positives and false negatives in that context.
* Run inference of the two models on the test set and print a table with the metrics to compare. 

In [ ]:
model_enb0 = efficientnet_b0()
model_enb0.classifier[1] = nn.Linear(1280, 2)
model_enb0.features[0][0] = nn.Conv2d(1, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
model_enb0.load_state_dict(torch.load('./efficientnetb0-pathology-cxr.pth'))

test_df = ... # COMPLETE
test_dataset_cls = ... # COMPLETE
test_dataloader = ... # COMPLETE

model_enb0.eval()
model_enb0.to(device)

preds_list = []
probs_list = []
label_list = []

with torch.no_grad():
    for X, y in test_dataloader:
        X, y = X.to(device), y.to(device)
        out = model_enb0(X)

        probs = nn.functional.softmax(out, dim=1)  # use softmax for multi-logit outputs
        preds = probs.argmax(dim=1)

        probs_list.append(probs[:, 1].item())  # probability for class 1
        preds_list.append(preds.item())
        label_list.append(y.detach().cpu().numpy())

display = RocCurveDisplay.from_predictions(
... # COMPLETE
name=f"Pneumonia vs. rest",
color="darkorange",
plot_chance_level=True,
)
_ = display.ax_.set(
xlabel="False Positive Rate",
ylabel="True Positive Rate",
title="Binary classification",
)
display.ax_.legend(frameon=False)

print(
f'Model EN-B0:',
'\nAccuracy:', ... # COMPLETE,
'\nF1-score:', ... # COMPLETE
'\nRecall:', ... # COMPLETE
'\nPrecision:', ... # COMPLETE
)

In [ ]:
model_unet = UNet(2,1,1)
model_unet.load_state_dict(torch.load('./unet_seg_cls_pathology_cxr.pth'))

test_df = ... # COMPLETE
test_dataset_cls = ... # COMPLETE
test_dataloader = ... # COMPLETE

model_unet.eval()
model_unet.to(device)

preds_list = []
probs_list = []
label_list = []

with torch.no_grad():
    for X, y in test_dataloader:
        X, y = X.to(device), y.to(device)
        out, _ = model_unet(X)

        probs = nn.functional.softmax(out, dim=1)  # use softmax for multi-logit outputs
        preds = probs.argmax(dim=1)

        probs_list.append(probs[:, 1].item())  # probability for class 1
        preds_list.append(preds.item())
        label_list.append(y.detach().cpu().numpy())

display = RocCurveDisplay.from_predictions(
... # COMPLETE
name=f"Pneumonia vs. rest",
color="darkorange",
plot_chance_level=True,
)
_ = display.ax_.set(
xlabel="False Positive Rate",
ylabel="True Positive Rate",
title="Binary classification",
)
display.ax_.legend(frameon=False)

print(
f'Model EN-B0:',
'\nAccuracy:', ... # COMPLETE
'\nF1-score:', ... # COMPLETE
'\nRecall:', ... # COMPLETE
'\nPrecision:', ... # COMPLETE
)